In [21]:
# Load env variables and create client
import base64
import os

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

client = Anthropic(
    base_url="https://openrouter.ai/api",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
model = "anthropic/claude-haiku-4.5"  # "deepseek/deepseek-v4.1-flash"

In [3]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [6]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure

4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [12]:
# TODO: Read image data, feed into Claude
with open("images/prop1.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []
add_user_message(
    messages,
    [
        # Image Block
        {
            "type": "image",
            "source": {
                "type": "base64",
                "media_type": "image/png",
                "data": image_bytes,
            },
        },
        # Text Block
        {
            "type": "text",
            "text": prompt,
        },
    ],
)

chat(messages, thinking=False)

Message(id='gen-1790660855-BFq441xB3Bhq9xJZOlVO', container=None, content=[ThinkingBlock(signature='', thinking='1.  **Analyze the Request:**\n    *   **Input:** A satellite image of a property.\n    *   **Task:** Analyze the image in 5 specific steps.\n    *   **Output Format:** One sentence per step (1-5), followed by a final numeric Fire Risk Rating (1-4) with a brief justification.\n\n2.  **Analyze the Image:**\n    *   *Step 1: Residence Identification.*\n        *   Look for the largest roofed structure. There\'s a prominent structure in the upper-left-center of the image. It has a light-colored roof (possibly grey/beige), some distinct geometry (an L-shape or complex roof), and seems to be surrounded by trees.\n        *   There\'s a dark, sharp triangular shape at the top of the roof (maybe a shadow or a dark roof section).\n        *   Driveway connection: Hard to tell, maybe a path at the top left or right.\n        *   Let\'s locate it relative to boundaries: The property se

In [22]:
import base64

with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",
            "source": {
                "type": "base64",
                "media_type": "application/pdf",
                "data": file_bytes,
            },
            "title": "Earth PDF",
            "citations": {"enabled": True},
        },
        {"type": "text", "text": "Summarize the document in one sentence, with one citation."},
    ],
)

reply = chat(messages)
# text_from_message(reply)

In [23]:
print(reply.model_dump_json(indent=2))

{
  "id": "gen-1790661403-dP5yFNchvSldCSesQMVk",
  "container": null,
  "content": [
    {
      "citations": [],
      "text": "# Summary\n\nEarth is the third planet from the Sun and the only known astronomical object to harbor life, enabled by its oceans covering 70.8% of its crust and a dynamic atmosphere composed primarily of nitrogen and oxygen.[1]",
      "type": "text"
    }
  ],
  "model": "anthropic/claude-haiku-4.5",
  "role": "assistant",
  "stop_details": null,
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": null,
    "input_tokens": 9630,
    "output_tokens": 52,
    "output_tokens_details": {
      "thinking_tokens": 0
    },
    "server_tool_use": null,
    "service_tier": "standard",
    "speed": "standard",
    "cost": 0.00989,
    "